# Phase 2: Data Cleaning

## 1. Introduction

Data cleaning is an essential step in preparing a reliable dataset
for machine learning and image captioning tasks.

The COCO dataset contains images, human-written captions,
and object annotations. Before using this information in our
Metadata-Filtered RAG for Image Captioning project, we must
verify the quality and consistency of the local data.

This phase focuses on identifying corrupted images,
detecting duplicate files, checking missing captions,
and preparing clean datasets.

## 2. Objectives

The main objectives of this phase are:

- Verify the integrity of local image files.
- Detect and identify corrupted images.
- Detect duplicate images.
- Check whether every image has associated captions.
- Clean and normalize caption text.
- Generate clean CSV files and image manifests.
- Document the cleaning operations and their results.

## 3. Dataset Description

The project uses a subset of the COCO 2017 dataset:

- Training dataset: 10,000 images.
- Validation dataset: 100 separate images.

The original COCO annotation files are stored in the
annotations directory.

## 4. Methodology

The cleaning process is performed in several steps:

1. Load the required libraries and define paths.
2. Verify the local image files.
3. Detect corrupted images.
4. Detect exact duplicate images using file hashes.
5. Associate images with their captions.
6. Clean caption text.
7. Export the cleaned datasets and generate a report.

## 5. Expected Output

At the end of this phase, we will obtain clean image files,
clean caption datasets, CSV manifests, and a report
documenting the cleaning operations.

## Step 1: Import Required Libraries

### Description
This step imports the Python libraries required for image
verification, duplicate detection, data manipulation,
and file management.

### Objective
Prepare the environment for the data cleaning process.

### Libraries
- Pandas: data manipulation.
- Pillow: image verification.
- Hashlib: duplicate detection.
- JSON: reading COCO annotations.
- Pathlib: file and directory management.
- Regular Expressions: text cleaning.

In [1]:
import json
import re
import hashlib
from pathlib import Path
from collections import Counter

import pandas as pd
from PIL import Image, UnidentifiedImageError

print("Libraries imported successfully!")

Libraries imported successfully!


## Step 2: Define Project Paths

### Description
We define the locations of the project folders and annotation files.

### Objective
Ensure that all cleaning operations use the correct directories.

### Method
We use pathlib to define paths in a clear and organized way.

In [2]:
PROJECT_DIR = Path(r"C:\Users\PC\Desktop\Metadata_RAG_START")

ANNOTATIONS_DIR = PROJECT_DIR / "annotations"
IMAGES_DIR = PROJECT_DIR / "images"

TRAIN_DIR = IMAGES_DIR / "train_images"
VALIDATION_DIR = IMAGES_DIR / "validation_images"

DATA_DIR = PROJECT_DIR / "data"
CLEANED_DIR = DATA_DIR / "cleaned"

CAPTIONS_PATH = ANNOTATIONS_DIR / "captions_train2017.json"

# Create output directories if they do not exist
DATA_DIR.mkdir(parents=True, exist_ok=True)
CLEANED_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR.exists())
print("Training directory:", TRAIN_DIR.exists())
print("Validation directory:", VALIDATION_DIR.exists())
print("Output directory:", CLEANED_DIR.exists())

Project directory: True
Training directory: True
Validation directory: True
Output directory: True


## Step 3: Collect Local Image Files

### Description
We collect the image files available in the Training
and Validation directories.

### Objective
Create a list of local images that will be checked
during the cleaning process.

### Method
We identify files with supported image extensions
such as JPG, JPEG, and PNG.

In [3]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

def get_image_files(folder):
    return sorted(
        file for file in folder.iterdir()
        if file.is_file() and file.suffix.lower() in IMAGE_EXTENSIONS
    )

train_files = get_image_files(TRAIN_DIR)
validation_files = get_image_files(VALIDATION_DIR)

print("Training images:", len(train_files))
print("Validation images:", len(validation_files))
print("Total local images:", len(train_files) + len(validation_files))

Training images: 10000
Validation images: 100
Total local images: 10100


## Step 4: Verify Image Integrity

### Description
Some image files may be corrupted or unreadable.
Such files can cause errors during image processing.

### Objective
Identify images that cannot be opened or verified.

### Method
We use the Pillow library to verify each image file.
Valid images are recorded, while corrupted images
are added to a separate list.

### Expected Output
A list of valid and corrupted image files.

In [4]:
def verify_images(image_files):
    valid_images = []
    corrupted_images = []

    for image_path in image_files:
        try:
            with Image.open(image_path) as img:
                img.verify()

            valid_images.append(image_path)

        except (UnidentifiedImageError, OSError, ValueError):
            corrupted_images.append(image_path)

    return valid_images, corrupted_images


valid_train, corrupted_train = verify_images(train_files)
valid_validation, corrupted_validation = verify_images(validation_files)

print("Training:")
print("Valid images:", len(valid_train))
print("Corrupted images:", len(corrupted_train))

print("\nValidation:")
print("Valid images:", len(valid_validation))
print("Corrupted images:", len(corrupted_validation))

Training:
Valid images: 10000
Corrupted images: 0

Validation:
Valid images: 100
Corrupted images: 0


## Step 5: Detect Duplicate Images

### Description
Duplicate images are repeated copies of the same image.
They may affect dataset quality and retrieval results.

### Objective
Identify exact duplicate images in the local dataset.

### Method
We calculate the SHA-256 hash of every valid image.
Images with identical hashes are considered exact duplicates.

We keep one copy of each duplicate group and record
the other copies for removal from the cleaned dataset.

The original image files are not deleted.

In [5]:
def calculate_hash(image_path):
    sha256 = hashlib.sha256()

    with open(image_path, "rb") as file:
        for chunk in iter(lambda: file.read(8192), b""):
            sha256.update(chunk)

    return sha256.hexdigest()


all_valid_images = valid_train + valid_validation

hash_groups = {}

for image_path in all_valid_images:
    image_hash = calculate_hash(image_path)

    hash_groups.setdefault(image_hash, []).append(image_path)


duplicate_groups = {
    image_hash: paths
    for image_hash, paths in hash_groups.items()
    if len(paths) > 1
}

duplicate_files = []

for paths in duplicate_groups.values():
    duplicate_files.extend(paths[1:])

print("Number of duplicate groups:", len(duplicate_groups))
print("Number of duplicate files:", len(duplicate_files))

Number of duplicate groups: 1
Number of duplicate files: 1


## Step 6: Prepare Clean Image Lists

### Description
After identifying corrupted and duplicate files,
we prepare lists containing only valid and unique images.

### Objective
Create clean Training and Validation image lists.

### Method
We exclude corrupted files and duplicate copies
without deleting the original files.

The datasets remain separate.

In [6]:
duplicate_paths = set(duplicate_files)

clean_train_files = [
    path for path in valid_train
    if path not in duplicate_paths
]

clean_validation_files = [
    path for path in valid_validation
    if path not in duplicate_paths
]

print("Clean Training images:", len(clean_train_files))
print("Clean Validation images:", len(clean_validation_files))

print(
    "Total clean images:",
    len(clean_train_files) + len(clean_validation_files)
)

Clean Training images: 9999
Clean Validation images: 100
Total clean images: 10099


## Step 7: Load COCO Captions

### Description
The COCO captions annotation file contains image identifiers,
file names, and human-written descriptions.

### Objective
Load the original captions and prepare them for association
with the local images.

### Method
We use the image ID to connect image information
with its corresponding captions.

In [7]:
with open(CAPTIONS_PATH, "r", encoding="utf-8") as file:
    captions_data = json.load(file)

print("Captions loaded successfully!")

print("Number of annotated images:", len(captions_data["images"]))
print("Number of captions:", len(captions_data["annotations"]))

Captions loaded successfully!
Number of annotated images: 118287
Number of captions: 591753


## Step 8: Associate Images with Captions

### Description
Each COCO image is identified by a unique image ID.
Captions are associated with images through this identifier.

### Objective
Build a mapping between image IDs and their captions.

### Method
We create a dictionary where each image ID is associated
with a list of its human-written captions.

In [8]:
image_info = {
    image["file_name"]: image
    for image in captions_data["images"]
}

captions_by_image = {}

for annotation in captions_data["annotations"]:
    image_id = annotation["image_id"]
    caption = annotation["caption"]

    captions_by_image.setdefault(image_id, []).append(caption)

print("Image information records:", len(image_info))
print("Images with captions:", len(captions_by_image))

Image information records: 118287
Images with captions: 118287


## Step 9: Check Missing Captions

### Description
Every image used in the captioning project should have
at least one associated human-written caption.

### Objective
Identify local images that do not have corresponding
caption annotations.

### Method
We compare local image file names with the image names
available in the COCO captions annotation file.

In [9]:
def check_missing_captions(image_files):
    missing = []
    available = []

    for image_path in image_files:
        if image_path.name not in image_info:
            missing.append(image_path.name)
            continue

        image_id = image_info[image_path.name]["id"]

        if image_id not in captions_by_image:
            missing.append(image_path.name)
        else:
            available.append(image_path.name)

    return available, missing


train_with_captions, train_without_captions = check_missing_captions(
    clean_train_files
)

validation_with_captions, validation_without_captions = check_missing_captions(
    clean_validation_files
)

print("Training images with captions:", len(train_with_captions))
print("Training images without captions:", len(train_without_captions))

print("\nValidation images with captions:", len(validation_with_captions))
print("Validation images without captions:", len(validation_without_captions))

Training images with captions: 9999
Training images without captions: 0

Validation images with captions: 100
Validation images without captions: 0


## Step 10: Clean Caption Text

### Description
Captions may contain unnecessary spaces or inconsistent formatting.

### Objective
Normalize caption text while preserving its original meaning.

### Method
We remove leading and trailing spaces, normalize repeated
whitespace, and remove empty captions.

In [10]:
def clean_caption(caption):
    caption = caption.strip()
    caption = re.sub(r"\s+", " ", caption)

    return caption


clean_caption_records = []

for image_name in train_with_captions + validation_with_captions:

    image_id = image_info[image_name]["id"]

    for caption in captions_by_image[image_id]:

        cleaned = clean_caption(caption)

        if cleaned:
            clean_caption_records.append({
                "image_id": image_id,
                "file_name": image_name,
                "caption": cleaned
            })


clean_captions_df = pd.DataFrame(clean_caption_records)

print("Clean caption records:", len(clean_captions_df))
display(clean_captions_df.head())

Clean caption records: 50521


,image_id,file_name,caption
0,36,000000000036.jpg,Woman in swim suit holding parasol on sunny day.
1,36,000000000036.jpg,"A woman posing for the camera, holding a pink,..."
2,36,000000000036.jpg,A woman in a floral swimsuit holds a pink umbr...
3,36,000000000036.jpg,A woman with an umbrella near the sea
4,36,000000000036.jpg,A girl in a bathing suit with a pink umbrella.


## Step 11: Separate Clean Captions

### Description
The cleaned captions are divided according to their
corresponding local image directories.

### Objective
Generate separate caption files for Training and Validation.

### Method
We use the local image file names to separate the records.
The two datasets remain independent.

In [11]:
train_names = set(train_with_captions)
validation_names = set(validation_with_captions)

train_captions_df = clean_captions_df[
    clean_captions_df["file_name"].isin(train_names)
].copy()

validation_captions_df = clean_captions_df[
    clean_captions_df["file_name"].isin(validation_names)
].copy()

train_captions_df.to_csv(
    DATA_DIR / "train_captions.csv",
    index=False,
    encoding="utf-8"
)

validation_captions_df.to_csv(
    DATA_DIR / "validation_captions.csv",
    index=False,
    encoding="utf-8"
)

print("Training captions saved:", len(train_captions_df))
print("Validation captions saved:", len(validation_captions_df))

Training captions saved: 50018
Validation captions saved: 503


## Step 12: Generate Image Manifests

### Description
A manifest is a structured file containing information
about the images included in a dataset.

### Objective
Create a record of the clean images and their corresponding IDs.

### Method
We generate separate manifests for Training and Validation,
including the image ID, file name, dimensions, and image path.

In [12]:
def create_manifest(image_names, dataset_name, folder):

    records = []

    for image_name in image_names:

        info = image_info[image_name]

        records.append({
            "dataset": dataset_name,
            "image_id": info["id"],
            "file_name": image_name,
            "width": info["width"],
            "height": info["height"],
            "image_path": str(folder / image_name)
        })

    return pd.DataFrame(records)


train_manifest = create_manifest(
    train_with_captions,
    "training",
    TRAIN_DIR
)

validation_manifest = create_manifest(
    validation_with_captions,
    "validation",
    VALIDATION_DIR
)


train_manifest.to_csv(
    DATA_DIR / "train_manifest.csv",
    index=False,
    encoding="utf-8"
)

validation_manifest.to_csv(
    DATA_DIR / "validation_manifest.csv",
    index=False,
    encoding="utf-8"
)

print("Training manifest:", len(train_manifest))
print("Validation manifest:", len(validation_manifest))

Training manifest: 9999
Validation manifest: 100


## Step 13: Data Cleaning Report

### Description
This step summarizes the results of the cleaning process.

### Objective
Document the number of valid, corrupted, duplicate,
and captioned images.

### Method
We collect the results obtained in the previous steps
and organize them into a summary table.

In [13]:
cleaning_report = pd.DataFrame([
    {
        "Dataset": "Training",
        "Original Images": len(train_files),
        "Valid Images": len(valid_train),
        "Corrupted Images": len(corrupted_train),
        "Duplicate Files": sum(
            path in duplicate_paths for path in train_files
        ),
        "Clean Images with Captions": len(train_with_captions),
        "Images without Captions": len(train_without_captions)
    },
    {
        "Dataset": "Validation",
        "Original Images": len(validation_files),
        "Valid Images": len(valid_validation),
        "Corrupted Images": len(corrupted_validation),
        "Duplicate Files": sum(
            path in duplicate_paths for path in validation_files
        ),
        "Clean Images with Captions": len(validation_with_captions),
        "Images without Captions": len(validation_without_captions)
    }
])

display(cleaning_report)

cleaning_report.to_csv(
    DATA_DIR / "cleaning_report.csv",
    index=False,
    encoding="utf-8"
)

print("Cleaning report saved successfully!")

,Dataset,Original Images,Valid Images,Corrupted Images,Duplicate Files,Clean Images with Captions,Images without Captions
0,Training,10000,10000,0,1,9999,0
1,Validation,100,100,0,0,100,0


Cleaning report saved successfully!


## Step 14: Inspect Duplicate Images

### Description
The duplicate detection process identified one group
of identical images.

### Objective
Identify the filenames and locations of duplicate images.

### Method
We display the paths of all images belonging to
duplicate groups to understand where the duplicates occur.

The original files will not be deleted.

In [14]:
# Display duplicate image details

for group_number, (image_hash, paths) in enumerate(
    duplicate_groups.items(), start=1
):
    print(f"\nDuplicate Group {group_number}")

    for path in paths:
        print("File:", path.name)
        print("Folder:", path.parent.name)
        print("Full path:", path)


Duplicate Group 1
File: 000000438867.jpg
Folder: train_images
Full path: C:\Users\PC\Desktop\Metadata_RAG_START\images\train_images\000000438867.jpg
File: 000000531721.jpg
Folder: train_images
Full path: C:\Users\PC\Desktop\Metadata_RAG_START\images\train_images\000000531721.jpg


## Step 15: Duplicate Detection Results

### Results

The duplicate detection process identified one group
containing two identical image files:

- 000000438867.jpg
- 000000531721.jpg

Both files are located in the Training directory.

### Analysis

The two files have identical SHA-256 hashes,
which confirms that they are exact duplicates.

No duplicates were detected between the Training
and Validation datasets.

### Cleaning Decision

One copy is retained in the cleaned Training dataset,
while the duplicate copy is excluded from the cleaned list.

The original image files are preserved.

In [15]:
# Display the retained and excluded duplicate files

for paths in duplicate_groups.values():
    retained_file = paths[0]
    excluded_files = paths[1:]

    print("Retained image:", retained_file.name)

    for path in excluded_files:
        print("Excluded duplicate:", path.name)

    print("Retained file exists:", retained_file.exists())

Retained image: 000000438867.jpg
Excluded duplicate: 000000531721.jpg
Retained file exists: True


## Step 16: Final Cleaning Report

### Description
This step summarizes the final results of the image
cleaning and caption verification processes.

### Objective
Provide an accurate overview of the Training and
Validation datasets after cleaning.

### Method
We calculate the number of valid images, corrupted images,
duplicate files, clean images with captions, and images
without captions.

In [19]:
cleaning_report = pd.DataFrame([
    {
        "Dataset": "Training",
        "Original Images": len(train_files),
        "Valid Images": len(valid_train),
        "Corrupted Images": len(corrupted_train),
        "Duplicate Files": sum(
            path in duplicate_paths for path in train_files
        ),
        "Clean Images": len(clean_train_files),
        "Images With Captions": len(train_with_captions),
        "Images Without Captions": len(train_without_captions)
    },
    {
        "Dataset": "Validation",
        "Original Images": len(validation_files),
        "Valid Images": len(valid_validation),
        "Corrupted Images": len(corrupted_validation),
        "Duplicate Files": sum(
            path in duplicate_paths for path in validation_files
        ),
        "Clean Images": len(clean_validation_files),
        "Images With Captions": len(validation_with_captions),
        "Images Without Captions": len(validation_without_captions)
    }
])

display(cleaning_report)

cleaning_report.to_csv(
    DATA_DIR / "cleaning_report.csv",
    index=False,
    encoding="utf-8"
)

print("Final cleaning report saved successfully!")

,Dataset,Original Images,Valid Images,Corrupted Images,Duplicate Files,Clean Images,Images With Captions,Images Without Captions
0,Training,10000,10000,0,1,9999,9999,0
1,Validation,100,100,0,0,100,100,0


Final cleaning report saved successfully!


## Step 17: Verify Generated Files

### Description
We verify that all expected CSV files were successfully
created during the cleaning process.

### Objective
Ensure that the cleaned captions, manifests, and report
are available for the next phase.

In [ ]:
expected_files = [
    DATA_DIR / "train_captions.csv",
    DATA_DIR / "validation_captions.csv",
    DATA_DIR / "train_manifest.csv",
    DATA_DIR / "validation_manifest.csv",
    DATA_DIR / "cleaning_report.csv"
]

for file_path in expected_files:
    print(
        file_path.name,
        "->",
        "Created" if file_path.exists() else "Missing"
    )

train_captions.csv -> Created
validation_captions.csv -> Created
train_manifest.csv -> Created
validation_manifest.csv -> Created
cleaning_report.csv -> Created


# Final Conclusion – Notebook 02

## 1. Summary

This notebook focused on cleaning and preparing the local
COCO Training and Validation datasets.

The following operations were performed:

- Image integrity verification.
- Corrupted image detection.
- Exact duplicate detection using SHA-256.
- Missing caption verification.
- Caption text normalization.
- Generation of clean caption CSV files.
- Creation of image manifests.
- Generation of a final cleaning report.

## 2. Cleaning Results

### Training Dataset
- Original images: 10,000.
- Valid images: 10,000.
- Corrupted images: 0.
- Duplicate files: 1.
- Clean images: 9,999.
- Images without captions: 0.

### Validation Dataset
- Original images: 100.
- Valid images: 100.
- Corrupted images: 0.
- Duplicate files: 0.
- Clean images: 100.
- Images without captions: 0.

## 3. Generated Files

The following files were successfully generated:

- train_captions.csv
- validation_captions.csv
- train_manifest.csv
- validation_manifest.csv
- cleaning_report.csv

## 4. Conclusion

The cleaning process identified one exact duplicate
in the Training dataset. One copy was retained in the
cleaned dataset, while the original files were preserved.

All remaining images were valid and had associated captions.

The cleaned datasets are now prepared for the next phase:
statistical analysis and data visualization.

Training and Validation remain separate throughout
the preprocessing process.